# trajectreview modeling notebook
この notebook は `COLMAP 4.0.x` と `nerfstudio splatfacto` による baseline route の Colab 実行雛形です。


In [ ]:
CONFIG = {
    'session_root': '/content/drive/MyDrive/trajectreview/input/replace-session-id',
    'input_root': '/content/drive/MyDrive/trajectreview/input',
    'work_root': '/content/trajectreview',
    'result_root': '/content/drive/MyDrive/trajectreview/results',
}
CONFIG


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json

session_root = Path(CONFIG['session_root'])
session_package = json.loads((session_root / 'session_package.json').read_text(encoding='utf-8'))
session_id = session_package['sessionId']
selected_route_path = session_root / 'selected_route.json'
job_request_path = session_root / 'colab_job_request.json'
route_id = 'route-colmap40-global-splatfacto'
mapper = 'global'
if selected_route_path.exists():
    selected_route = json.loads(selected_route_path.read_text(encoding='utf-8'))
    route_id = selected_route.get('selectedRouteId', route_id)
    mapper = selected_route.get('selectedRoute', {}).get('mapper', mapper)
elif job_request_path.exists():
    job_request = json.loads(job_request_path.read_text(encoding='utf-8'))
    route_id = job_request.get('defaultRouteId', route_id)
images_source = session_root / 'images'
required = [
    session_root / 'video.mp4',
    session_root / 'session_package.json',
    session_root / 'frame_pose_index.csv',
    session_root / 'sensor_quality.json',
    session_root / 'space_handoff_manifest.json',
    images_source,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'missing inputs: {missing}')
session_root, session_id, route_id, mapper


In [ ]:
import subprocess

def run(cmd):
    print('RUN', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run(['bash', '-lc', 'apt-get update'])
run(['bash', '-lc', 'apt-get install -y colmap ffmpeg'])
run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'])
run(['python', '-m', 'pip', 'install', 'nerfstudio'])


In [ ]:
from pathlib import Path
work_root = Path(CONFIG['work_root'])
images_dir = work_root / 'images'
db_path = work_root / 'colmap.db'
sparse_dir = work_root / 'sparse'
processed_dir = work_root / 'processed'
export_dir = Path(CONFIG['result_root']) / session_id / route_id
for directory in [work_root, images_dir, sparse_dir, processed_dir, export_dir]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import shutil
for image_path in images_source.iterdir():
    if image_path.is_file():
        shutil.copy2(image_path, images_dir / image_path.name)
print({'copied_images': len(list(images_dir.iterdir()))})


In [ ]:
run(['colmap', 'feature_extractor', '--database_path', str(db_path), '--image_path', str(images_dir), '--ImageReader.single_camera', '1'])
run(['colmap', 'exhaustive_matcher', '--database_path', str(db_path)])
mapper_command = {
    'incremental': ['colmap', 'mapper', '--database_path', str(db_path), '--image_path', str(images_dir), '--output_path', str(sparse_dir)],
    'hierarchical': ['colmap', 'hierarchical_mapper', '--database_path', str(db_path), '--image_path', str(images_dir), '--output_path', str(sparse_dir)],
    'global': ['colmap', 'global_mapper', '--database_path', str(db_path), '--image_path', str(images_dir), '--output_path', str(sparse_dir)],
}[mapper]
run(mapper_command)
run(['ns-process-data', 'images', '--data', str(images_dir), '--output-dir', str(processed_dir)])
run(['ns-train', 'splatfacto', '--data', str(processed_dir), '--output-dir', str(export_dir)])


In [ ]:
summary = {
    'sessionId': session_id,
    'routeId': route_id,
    'mapper': mapper,
    'status': 'completed',
}
(export_dir / 'remote_summary.json').write_text(__import__('json').dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
